# 45 · Checkpoint: benchmark and tune Scout
Level L4 · Part 5 · professional · ~90 min · builds on 35, 37, 40

## Goal
Run Scout on 10 fixed questions in four configurations (two Gemini models × two reasoning-effort
levels), print one table of success rate, steps, tokens, cost and latency, and pick a configuration with
a written rule. The pick ends in a one-paragraph recommendation built from the measured numbers.

## The idea
"Which model and which effort level?" is a measurement question. A **benchmark** holds everything fixed
except the thing you are choosing: the same questions, the same tools, the same loop, the same scorer.
Only the **configuration** (model + effort) changes. Each run records what the business pays for:
right answers, tokens, dollars and seconds. Then a rule decides, written down *before* you look:
first a quality floor, then a latency budget, then the lowest cost.

Part 5 gave you the parts. Lesson 37's adapter lets one loop run on any model. Lesson 40 showed that
effort changes thinking tokens, cost and time.

Analogy: test-driving cars for a delivery fleet. Every car drives the same route with the same load,
and you choose by the fleet's rule (must arrive on time, then cheapest fuel), not by which looks fastest.

## Picture
```mermaid
flowchart LR
    B["10 fixed questions<br/>+ is_correct"] --> R["run_config<br/>(same loop, same tools)"]
    C["4 configs<br/>model × effort"] --> R
    R --> U["per question:<br/>correct, steps, tokens, seconds"]
    U --> T["summary table"]
    T --> P["rule: floor → latency → cost"]
    P --> REC["recommendation<br/>with numbers"]
```

## Requirements
- [ ] a `ScoutModel(model, level)` adapter with lesson 37's call shape, `model(messages, schemas)`, that
  sets Gemini's `thinking_level` (lesson 40) and appends one usage row per call to `self.calls`:
  `{"in", "out", "thoughts", "seconds"}`;
- [ ] `run_config(model)` that runs all 10 `BENCH` questions through `run_agent` on **one fresh**
  adapter and returns one row per question: `question`, `correct`, `answer`, `steps`, `errors`, `stop`,
  `calls` (that question's usage rows only). A crash in one question is a failed row, not a crash;
- [ ] `summarise(rows, model)`: success rate, mean steps, mean input / output / thinking tokens per
  question, cost per question (thinking billed as output, **each model at its own price**), and the
  p50 and p95 seconds per question;

- [ ] the four configurations run on the real model, with their timings kept from the recording run;
- [ ] `recommend(table, floor, budget_s)` that returns the cheapest configuration that meets the
  quality floor and the latency budget, and a one-paragraph recommendation that quotes the numbers.

Done when `check_bench(run_config)` prints `all checks passed` on the fake model, and the table and
the recommendation print from real (recorded) runs.

## Build it
### Starter code: the recorder, the client, the configurations and the prices
Every Gemini call is recorded once to `../data/llm_cassettes/45_checkpoint-benchmark-scout.yaml` and
replayed after that. Prices are per million tokens, from RESEARCH.md (Gemini pricing page, checked
2026-10-06); `gemini-3.8-flash` is on an introductory price until 2026-12-31 (lesson 17).

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("45_checkpoint-benchmark-scout")

import ast, json, operator, os, pathlib, time
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
CONFIGS = [("gemini-3.5-flash-lite", "LOW"), ("gemini-3.5-flash-lite", "HIGH"),
           ("gemini-3.8-flash", "LOW"), ("gemini-3.8-flash", "HIGH")]
PRICES_PER_MTOK = {"gemini-3.5-flash-lite": {"input": 0.30, "output": 2.50},
                   "gemini-3.8-flash":      {"input": 0.75, "output": 3.75}}   # USD, 2026-10-06

def cost_usd(model: str, input_tokens: int, output_tokens: int) -> float:
    price = PRICES_PER_MTOK[model]
    return (input_tokens * price["input"] + output_tokens * price["output"]) / 1_000_000

print(len(CONFIGS), "configurations:", [f"{m}/{lv}" for m, lv in CONFIGS])

LLM mode: replay (45_checkpoint-benchmark-scout.yaml)


4 configurations: ['gemini-3.5-flash-lite/LOW', 'gemini-3.5-flash-lite/HIGH', 'gemini-3.8-flash/LOW', 'gemini-3.8-flash/HIGH']


### Starter code: Scout's notes and read-only tools (lesson 35, condensed as in lesson 40)
The six facts, the long briefing and the 30-day habitat log; `search_notes` with pages, and the safe
`calculator`. The benchmark needs no `write_file`, so the runs leave no files behind.

In [2]:
NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
BRIEFING = {"id": 7, "text": "Mars mission briefing: " + "check suits, check air, check water. " * 40}
LOG = [{"id": 100 + d, "text": f"Mars habitat log, day {d}: routine checks, all systems normal."} for d in range(1, 31)]
ALL_NOTES = NOTES + [BRIEFING] + LOG

def search_notes(query: str, max_results: int = 5, page: int = 1) -> dict:
    q = query.strip().lower()
    if not q or page < 1:
        raise ValueError("query must not be empty and page starts at 1, e.g. query='Mars', page=1")
    hits = [n for n in ALL_NOTES if q in n["text"].lower()]
    size = min(max(max_results, 1), 10)
    start = (page - 1) * size
    return {"total": len(hits), "page": page,
            "results": [{"id": n["id"], "text": n["text"][:120]} for n in hits[start:start + size]],
            "next_page": page + 1 if start + size < len(hits) else None}

print(search_notes("Mars", max_results=2)["total"], "notes mention Mars")

34 notes mention Mars


The calculator walks the syntax tree with an allow-list of operations (lesson 29), and the two schemas
are written out by hand.

In [3]:
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.Mod: operator.mod, ast.Pow: operator.pow, ast.FloorDiv: operator.floordiv}
def evaluate(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return -evaluate(node.operand)
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        return OPS[type(node.op)](evaluate(node.left), evaluate(node.right))
    raise ValueError("not plain arithmetic: use numbers, + - * / // % ** and brackets, e.g. '24*60+39'")

def calculator(expression: str) -> float:
    return evaluate(ast.parse(expression, mode="eval").body)

SCHEMAS = [
    {"name": "search_notes", "description": "Search the user's notes for a word or phrase (case-insensitive). "
     "Returns one page of {id, text}, the total number of matches and next_page.",
     "parameters": {"type": "object", "required": ["query"], "properties": {
         "query": {"type": "string"}, "max_results": {"type": "integer"}, "page": {"type": "integer"}}}},
    {"name": "calculator", "description": "Evaluate arithmetic: numbers, + - * / // % ** and brackets.",
     "parameters": {"type": "object", "required": ["expression"], "properties": {"expression": {"type": "string"}}}},
]
TOOLS = {"search_notes": search_notes, "calculator": calculator}
print(calculator("1000 // 24.65"))

40.0


### Starter code: `run_tool` and `run_agent` (lesson 35)
`run_tool` never raises. `run_agent` is lesson 35's loop with the trace off: a benchmark prints one
line per question, not one per step.

In [4]:
def run_tool(call: dict, tools: dict) -> dict:
    fn = tools.get(call["name"])
    if fn is None:
        return {"content": f"unknown tool '{call['name']}'. Available: {', '.join(tools)}.", "is_error": True}
    try:
        return {"content": json.dumps(fn(**call["args"])), "is_error": False}
    except Exception as e:
        return {"content": f"{type(e).__name__}: {e}", "is_error": True}

def run_agent(question: str, model, tools: dict, schemas: list[dict], max_steps: int = 8) -> dict:
    messages, errors = [{"role": "user", "text": question}], 0
    for step in range(1, max_steps + 1):
        reply = model(messages, schemas)
        messages.append(reply)
        if not reply.get("tool_calls"):
            return {"answer": reply["text"], "stop": "end_of_turn", "steps": step, "errors": errors,
                    "messages": messages}
        for call in reply["tool_calls"]:
            result = run_tool(call, tools)
            errors += result["is_error"]
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"], **result})
    return {"answer": None, "stop": "step_limit", "steps": max_steps, "errors": errors, "messages": messages}

### Starter code: the 10 fixed questions and the scorer
Each question comes with the strings a right answer must contain. Most need a search and a
calculation; two need the search's `total`, not just its first page. The scorer removes thousands
commas first, because lesson 37 saw Gemini write "1,479".

In [5]:
BENCH = [
    ("How many minutes long is a day on Mars?", ["1479"]),
    ("What are the names of the moons of Mars?", ["Phobos", "Deimos"]),
    ("How many minutes longer is a day on Mars than a day on Earth?", ["39"]),
    ("How many seconds are there in 3 Mars days?", ["266220"]),
    ("Which planet spins backwards?", ["Venus"]),
    ("How many habitat log entries do I have?", ["30"]),
    ("How many of my notes mention Mars?", ["34"]),
    ("How many whole Mars days fit in 1000 Earth hours?", ["40"]),
    ("In what year did Perseverance land, and how many years before 2030 is that?", ["2021", "9"]),
    ("How many hours are 10 Mars days?", ["246.5"]),
]
def is_correct(answer: str | None, expected: list[str]) -> bool:
    text = (answer or "").replace(",", "").lower()
    return all(e.lower() in text for e in expected)

print(len(BENCH), "questions |", is_correct("A Mars day is 1,479 minutes.", ["1479"]))

10 questions | True


### Starter code: a fake Scout and the checks
Test the harness before you spend money on it. `FakeScout` has lesson 37's call shape: it searches
once, then answers from a table. It answers question 4 wrongly on purpose and charges fixed token
counts, so every number the harness reports is known in advance.

In [6]:
class FakeScout:
    name, model = "fake", "gemini-3.5-flash-lite"           # priced like flash-lite for the check
    def __init__(self):
        self.calls, self.answers = [], {q: " and ".join(exp) for q, exp in BENCH}
        self.answers[BENCH[3][0]] = "I don't know."          # one wrong answer on purpose

    def __call__(self, messages: list[dict], schemas: list[dict]) -> dict:
        self.calls.append({"in": 100, "out": 10, "thoughts": 50, "seconds": 0.5})
        if len(messages) == 1:                                # first step: search
            return {"role": "assistant", "tool_calls": [{"id": "c1", "name": "search_notes",
                                                         "args": {"query": "Mars"}}]}
        return {"role": "assistant", "text": self.answers[messages[0]["text"]]}

`check_bench` runs your `run_config` on the fake and checks every number the table will use.

In [7]:
def check_bench(run_config) -> None:
    rows = run_config(FakeScout())
    assert len(rows) == len(BENCH), f"expected one row per question, got {len(rows)}"
    assert sum(r["correct"] for r in rows) == 9, "the fake gets exactly 9 of 10 right"
    assert all(r["steps"] == 2 and len(r["calls"]) == 2 for r in rows), "each question: 2 steps, 2 calls"
    s = summarise(rows, "gemini-3.5-flash-lite")
    assert s["success"] == 0.9 and s["steps"] == 2.0, s
    assert abs(s["cost"] - cost_usd("gemini-3.5-flash-lite", 200, 120)) < 1e-12, "thinking billed as output"
    assert s["p50_s"] == s["p95_s"] == 1.0, "two 0.5 s calls per question"
    print("all checks passed")

### Your turn
Write `ScoutModel`, `run_config`, `summarise` and `recommend`. Pass `check_bench(run_config)`, then run
the four configurations on Gemini and write your recommendation. Lessons 37 and 40 have most of the
code you need.

In [8]:
# your code here

## Reference solution
Try it yourself first. The steps below are one way to meet every requirement.

### R1. The Gemini adapter with an effort level (lessons 37 and 40)
The request half turns neutral messages into Gemini contents; a model turn goes back as the `raw`
content Gemini returned, so thought signatures survive (lesson 40).

In [9]:
def to_contents(messages: list[dict]) -> list[types.Content]:
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])                          # keeps thought signatures
        else:
            response = {"error": m["content"]} if m["is_error"] else {"output": json.loads(m["content"])}
            part = types.Part(function_response=types.FunctionResponse(id=m["call_id"], name=m["name"],
                                                                       response=response))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

`timed` is lesson 40's clock: while recording, it waits out the recorder's 4-second spacing first, so
the time measures only the model.

In [10]:
def timed(call):
    if not llm_replay.is_replaying():           # recording or live: skip llm_replay's spacing
        time.sleep(float(os.environ.get("TUT_LLM_MIN_INTERVAL", "4")))
    t0 = time.perf_counter()
    result = call()
    return result, time.perf_counter() - t0

SYSTEM = ("You are Scout. Answer questions about the user's notes. Search the notes first. "
          "Use the calculator for arithmetic. If a tool returns an error, fix the call and try again. "
          "Answer in one sentence.")
print(SYSTEM[:40])

You are Scout. Answer questions about th


The reply half. The differences from lesson 37's `GeminiAdapter` are `level`, a 2-minute per-call
timeout (a smoke test saw one `HIGH` call take a minute), and the usage row, which now includes
thinking tokens and seconds.

In [11]:
class ScoutModel:
    def __init__(self, model: str, level: str, max_output_tokens: int = 6000):
        self.model, self.level, self.max_output_tokens = model, level, max_output_tokens
        self.name, self.calls = f"{model}/{level}", []      # one usage row per model call

    def __call__(self, messages: list[dict], schemas: list[dict]) -> dict:
        decls = [types.FunctionDeclaration(name=s["name"], description=s["description"],
                                           parameters_json_schema=s["parameters"]) for s in schemas]
        cfg = types.GenerateContentConfig(system_instruction=SYSTEM, max_output_tokens=self.max_output_tokens,
            tools=[types.Tool(function_declarations=decls)], automatic_function_calling=NO_AFC,
            thinking_config=types.ThinkingConfig(thinking_level=self.level),
            http_options=types.HttpOptions(timeout=120_000))            # 2 min per call (lesson 30)
        resp, seconds = timed(lambda: client.models.generate_content(
            model=self.model, contents=to_contents(messages), config=cfg))
        u = resp.usage_metadata
        self.calls.append({"in": u.prompt_token_count or 0, "out": u.candidates_token_count or 0,
                           "thoughts": u.thoughts_token_count or 0, "seconds": seconds})
        raw = resp.candidates[0].content
        calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp.function_calls or []]
        if calls:
            return {"role": "assistant", "tool_calls": calls, "raw": raw}
        return {"role": "assistant", "text": resp.text or "", "raw": raw}

### R2. `run_config`: one adapter, ten questions
A question's usage rows are the slice of `model.calls` it added. An exception (a timeout, a 500) is
recorded as a failed row, so one bad call does not throw away the other nine results.

In [12]:
def run_config(model) -> list[dict]:
    rows = []
    for question, expected in BENCH:
        first = len(model.calls)
        try:
            run = run_agent(question, model, TOOLS, SCHEMAS)
        except Exception as e:
            print(f"  question failed: {type(e).__name__}")
            run = {"answer": None, "stop": "error", "steps": len(model.calls) - first, "errors": 0}
        rows.append({"question": question, "correct": is_correct(run["answer"], expected),
                     "answer": run["answer"], "steps": run["steps"], "errors": run["errors"],
                     "stop": run["stop"], "calls": model.calls[first:]})
    print(f"{getattr(model, 'name', '?'):28} {sum(r['correct'] for r in rows)}/{len(rows)} right")
    return rows

### R3. `summarise`: one row of the table
Latency is per **question** (the sum of its calls), because that is what a user waits for. The p95 uses
the nearest-rank method; with 10 questions it is the slowest question, which is the honest reading.

In [13]:
def percentile(values: list[float], p: float) -> float:
    ordered = sorted(values)
    rank = int(-(-len(ordered) * p // 100))                        # nearest rank: ceil(n*p/100)
    return ordered[max(rank, 1) - 1]

def summarise(rows: list[dict], model: str) -> dict:
    calls, n = [c for r in rows for c in r["calls"]], len(rows)
    tin, tout, tthink = (sum(c[k] for c in calls) for k in ("in", "out", "thoughts"))
    per_q = [sum(c["seconds"] for c in r["calls"]) for r in rows]
    return {"success": sum(r["correct"] for r in rows) / n, "steps": sum(r["steps"] for r in rows) / n,
            "in": tin / n, "out": tout / n, "thinking": tthink / n,
            "cost": cost_usd(model, tin, tout + tthink) / n,             # thinking billed as output
            "p50_s": percentile(per_q, 50), "p95_s": percentile(per_q, 95)}

print(percentile([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 50), percentile([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 95))

5 10


### R4. Check the harness on the fake
No key, no cost: if this fails, the problem is in your harness, not in the model.

In [14]:
check_bench(run_config)

fake                         9/10 right
all checks passed


### R5. The real runs
Each configuration gets a **fresh** `ScoutModel`. A `results` dict collects the rows, keyed by name.

In [15]:
results = {}
print("to run:", [f"{m}/{lv}" for m, lv in CONFIGS])

to run: ['gemini-3.5-flash-lite/LOW', 'gemini-3.5-flash-lite/HIGH', 'gemini-3.8-flash/LOW', 'gemini-3.8-flash/HIGH']


The first two use `gemini-3.5-flash-lite`.

In [16]:
for model_id, level in CONFIGS[:2]:
    results[f"{model_id}/{level}"] = run_config(ScoutModel(model_id, level))

⏭  Skipped: this cell needs [slow]. Run it yourself, or re-run with TUT_ENABLE=slow.


Then `gemini-3.8-flash`, the larger Flash model, at both levels. Lesson 17 saw it take over a minute on
some calls, so this cell can be slow while recording.

In [17]:
for model_id, level in CONFIGS[2:]:
    results[f"{model_id}/{level}"] = run_config(ScoutModel(model_id, level))

⏭  Skipped: this cell needs [slow]. Run it yourself, or re-run with TUT_ENABLE=slow.


### R6. Keep the real timings
A replay answers instantly, so its seconds mean nothing. As in lesson 40, `keep_recorded` saves the
recording run's rows to `../data/recorded/45.json`. On a replay it puts the recorded seconds back; if
the model cells were skipped, it supplies the whole recorded result (**recorded output**).

In [18]:
RECORDED = pathlib.Path("../data/recorded/45.json")

def keep_recorded(fresh: dict) -> dict:
    complete = len(fresh) == len(CONFIGS)
    if complete and not llm_replay.is_replaying():               # recording run: save real numbers
        RECORDED.write_text(json.dumps(fresh))
    saved = json.loads(RECORDED.read_text())
    if not complete:
        return saved                                              # model cells skipped: recorded output
    for name, rows in fresh.items():                              # replay: restore the real seconds
        for row, old in zip(rows, saved[name]):
            for call, old_call in zip(row["calls"], old["calls"]):
                call["seconds"] = old_call["seconds"]
    return fresh

results = keep_recorded(results)
print({name: sum(r["correct"] for r in rows) for name, rows in results.items()})

{'gemini-3.5-flash-lite/LOW': 9, 'gemini-3.5-flash-lite/HIGH': 10, 'gemini-3.8-flash/LOW': 10, 'gemini-3.8-flash/HIGH': 8}


### R7. The table
One line per configuration. Token counts and cost are **per question**; seconds are per question too.

In [19]:
table = {name: summarise(rows, name.split("/")[0]) for name, rows in results.items()}
print(f"{'configuration':28} {'right':>5} {'steps':>5} {'in':>6} {'out':>4} {'think':>6} "
      f"{'$/question':>10} {'p50 s':>6} {'p95 s':>6}")
for name, s in table.items():
    print(f"{name:28} {s['success']:>5.0%} {s['steps']:>5.1f} {s['in']:>6.0f} {s['out']:>4.0f} "
          f"{s['thinking']:>6.0f} {s['cost']:>10.6f} {s['p50_s']:>6.1f} {s['p95_s']:>6.1f}")

configuration                right steps     in  out  think $/question  p50 s  p95 s
gemini-3.5-flash-lite/LOW      90%   3.3    750   68      0   0.000395    2.2    3.9
gemini-3.5-flash-lite/HIGH    100%   3.2   1359   71    438   0.001681    2.9    6.4
gemini-3.8-flash/LOW          100%   2.9    726   60     29   0.000877    4.6   10.8
gemini-3.8-flash/HIGH          80%   4.0   1779   80    404   0.003148    9.2  100.8


Reading the recorded run: `HIGH` effort did not buy accuracy. Flash-lite went from 9 to 10 right at
about 4x the cost, and `gemini-3.8-flash/HIGH` scored *lowest* (80%) at the highest cost. Thinking
tokens are the gap: about 400 per question at `HIGH`, near 0 at `LOW`. The `gemini-3.8-flash/HIGH`
p95 of 100.8 s is one question whose first call took 89 s. The recording stores only the total seconds
per call, so why that call was slow is unknown from this data; whatever the cause, a user waits
through it, so it counts.

### R8. Which questions failed, where
A success rate hides *which* questions failed. One line per question, one column per configuration.

In [20]:
names = list(results)
print(" ".join(f"{i + 1:>2}" for i in range(len(names))), " question")
for qi, (question, _) in enumerate(BENCH):
    marks = ["ok" if results[n][qi]["correct"] else "--" for n in names]
    print(" ".join(f"{m:>2}" for m in marks), "", question[:60])
for i, n in enumerate(names):
    print(f"{i + 1} = {n}")

 1  2  3  4  question
ok ok ok ok  How many minutes long is a day on Mars?
ok ok ok ok  What are the names of the moons of Mars?
ok ok ok ok  How many minutes longer is a day on Mars than a day on Earth
-- ok ok ok  How many seconds are there in 3 Mars days?
ok ok ok ok  Which planet spins backwards?
ok ok ok ok  How many habitat log entries do I have?
ok ok ok --  How many of my notes mention Mars?
ok ok ok --  How many whole Mars days fit in 1000 Earth hours?
ok ok ok ok  In what year did Perseverance land, and how many years befor
ok ok ok ok  How many hours are 10 Mars days?
1 = gemini-3.5-flash-lite/LOW
2 = gemini-3.5-flash-lite/HIGH
3 = gemini-3.8-flash/LOW
4 = gemini-3.8-flash/HIGH


Before trusting a "--", read the answer: a failure can be the model's or the scorer's.

In [21]:
for n in names:
    for row in results[n]:
        if not row["correct"]:
            print(f"{n}: {row['question'][:45]}\n    -> {str(row['answer'])[:110]!r} (stop: {row['stop']})")

gemini-3.5-flash-lite/LOW: How many seconds are there in 3 Mars days?
    -> 'There are approximately 266,326 seconds in 3 Mars days (sols), assuming a Mars day is about 24 hours, 39 minut' (stop: end_of_turn)
gemini-3.8-flash/HIGH: How many of my notes mention Mars?
    -> 'Thirty-four of your notes mention Mars.' (stop: end_of_turn)
gemini-3.8-flash/HIGH: How many whole Mars days fit in 1000 Earth ho
    -> 'None' (stop: step_limit)


Three failures, three different causes. Flash-lite `LOW` did the arithmetic in its head and got 266,326
instead of 266,220: a real model error. `gemini-3.8-flash/HIGH` answered "Thirty-four", which is right,
but the scorer looks for the digits "34": a **scorer** error. Its third failure hit the 8-step limit.
Fixing the scorer would lift that configuration to 90%, but it would still fail the latency budget.

### R9. The rule, then the pick
Write the rule before reading the table: the answer must be right at least 9 times in 10 (`floor`),
the slowest question must finish within 10 seconds (`budget_s`), and among the configurations that
pass both, the cheapest wins. If none pass, `recommend` says so instead of guessing.

In [22]:
def recommend(table: dict, floor: float = 0.9, budget_s: float = 10.0) -> str | None:
    passing = {n: s for n, s in table.items() if s["success"] >= floor and s["p95_s"] <= budget_s}
    for n, s in table.items():
        why = [w for w, bad in (("below floor", s["success"] < floor), ("too slow", s["p95_s"] > budget_s)) if bad]
        print(f"{n:28} {'passes' if not why else ', '.join(why)}")
    return min(passing, key=lambda n: (table[n]["cost"], table[n]["p50_s"])) if passing else None

best = recommend(table)
print("pick:", best)

gemini-3.5-flash-lite/LOW    passes
gemini-3.5-flash-lite/HIGH   passes
gemini-3.8-flash/LOW         too slow
gemini-3.8-flash/HIGH        below floor, too slow
pick: gemini-3.5-flash-lite/LOW


`gemini-3.8-flash/LOW` misses by 0.8 s on one question; with one run, treat it as a near miss, not a
clear loss.

### R10. The recommendation, with numbers
The paragraph is an f-string over the table, so it can never disagree with the measurements. Volume:
5,000 questions a day, 30 days.

In [23]:
def monthly(name: str, per_day: int = 5000) -> float:
    return table[name]["cost"] * per_day * 30

dearest = max(table, key=lambda n: table[n]["cost"])
if best is None:
    print("No configuration met the floor and the budget; widen the budget or improve the prompt and tools.")
else:
    b, d = table[best], table[dearest]
    print(f"Recommendation: run Scout on {best}. On the 10-question benchmark it answered {b['success']:.0%} "
          f"correctly in {b['steps']:.1f} model calls per question, at ${b['cost']:.6f} per question and "
          f"{b['p50_s']:.1f} s median ({b['p95_s']:.1f} s slowest). That is about ${monthly(best):,.0f} a month "
          f"at 5,000 questions a day. The dearest configuration, {dearest}, scored {d['success']:.0%} at "
          f"{d['cost'] / b['cost']:.1f}x the cost (${monthly(dearest):,.0f} a month) and {d['p50_s']:.1f} s "
          f"median. Caveat: one run of 10 questions; repeat each question several times before relying on "
          f"a difference of one answer.")

Recommendation: run Scout on gemini-3.5-flash-lite/LOW. On the 10-question benchmark it answered 90% correctly in 3.3 model calls per question, at $0.000395 per question and 2.2 s median (3.9 s slowest). That is about $59 a month at 5,000 questions a day. The dearest configuration, gemini-3.8-flash/HIGH, scored 80% at 8.0x the cost ($472 a month) and 9.2 s median. Caveat: one run of 10 questions; repeat each question several times before relying on a difference of one answer.


## At work
- **Configuration is chosen per route, with evals, not once per app.** Teams score each candidate
  model and effort level on a golden set, and send easy requests to the cheap configuration and hard
  ones to a bigger one. → lessons 48 and 54
- **One run per question is not enough.** Anthropic's agent-evals guide recommends several trials per
  task and reading the transcripts, because a sampling model can pass once and fail the next time;
  pass^k measures how often it passes every time. → lesson 48
- **Prices and models move under a decision.** `gemini-3.8-flash` doubles its price on 2027-01-01, and
  vendors retire model ids; teams keep the benchmark and re-run it before a price change or migration.
  → lesson 53
- **Dashboards must count thinking tokens.** OpenTelemetry's GenAI conventions log reasoning tokens
  separately (`gen_ai.usage.reasoning.output_tokens`); a cost report without them under-states the
  high-effort rows of this table. → lesson 49

## What just happened
- You held the questions, tools, loop and scorer fixed and changed only the configuration, so the
  differences are not caused by changed questions, tools or scorer. With one run of 10 questions, a
  one-answer gap can still be noise.
- The fake model tested the harness first: every number it reports was known in advance.
- Each configuration got a fresh adapter; cost used each model's own price, with thinking billed as output.
- The rule (90% floor, 10 s p95, then cheapest) was written before reading the table. It picked
  `gemini-3.5-flash-lite/LOW`: about $59 a month at 5,000 questions a day, against $472 for the dearest.
- Reading the failures showed that one "wrong" answer was the scorer's fault, not the model's.

## Common mistakes
**1. One adapter shared by every configuration.** `self.calls` keeps growing, so the second
configuration's slices are right but a total taken from `model.calls` includes the first
configuration's calls too. The fake makes it visible with no key.

In [24]:
shared = FakeScout()
first_rows = run_config(shared)
second_rows = run_config(shared)                             # same object again: calls pile up
second_total = len(shared.calls)
assert second_total == sum(r["steps"] for r in second_rows), \
    f"the second config ran {sum(r['steps'] for r in second_rows)} calls but the adapter counts {second_total}"

fake                         9/10 right
fake                         9/10 right


AssertionError: the second config ran 20 calls but the adapter counts 40

Fix: one fresh adapter per configuration, as R5 does, and take totals from the rows, not the adapter.

In [25]:
fresh = FakeScout()
rows = run_config(fresh)
print(len(fresh.calls) == sum(r["steps"] for r in rows))

fake                         9/10 right
True


**2. Pricing every configuration at one model's price.** Copying lesson 40's `cost_usd(MODEL, ...)`
prices `gemini-3.8-flash` at flash-lite rates and makes the larger model look cheap. A check against
the per-model price catches it.

In [26]:
MODEL = "gemini-3.5-flash-lite"
name = "gemini-3.8-flash/LOW"
wrong = summarise(results[name], MODEL)["cost"]               # every config at flash-lite prices
right = summarise(results[name], name.split("/")[0])["cost"]
assert wrong == right, f"{name}: priced at ${wrong:.6f} per question, really ${right:.6f}"

AssertionError: gemini-3.8-flash/LOW: priced at $0.000440 per question, really $0.000877

Fix: take the model id from the configuration's name (or store it in each row), as R7 does.

## Try it
No new model calls: use `results` and `table` from above. *Oracle routing* means sending each question
to the cheapest configuration that answered it correctly, as if a perfect router knew in advance.
1. From 2027-01-01, `gemini-3.8-flash` costs $1.50 / $7.50 per million tokens. Rebuild the table at the
   new price and run `recommend` again. Does the pick change? What is the new monthly cost of each
   `gemini-3.8-flash` configuration?
2. **Oracle routing.** Suppose a perfect router sent each question to the cheapest configuration that
   answered it correctly. Compute that routed cost per question and success rate, and compare them with
   the single recommended configuration. How much could routing (lesson 54) save at most here?

In [27]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [28]:
llm_replay.stop()

## Key terms
- **benchmark** — a fixed set of questions, a fixed scorer and a fixed harness, run once per configuration so that only the configuration changes between runs.
- **configuration** — one choice of the settings under test, here a model id plus a reasoning-effort level, written `model/LEVEL`.
- **quality floor** — the minimum success rate a configuration must reach before its cost or speed counts; set before looking at the results.
- **latency budget** — the longest a user may wait for one answer; here the benchmark's p95 seconds per question must stay within it.
- **oracle routing** — the cost of sending each question to the cheapest configuration that got it right; an upper bound on what routing could save.